# Case-Based Reasoning (CBR)
## Tahap 1 – Membangun Case Base (Scraping & Cleaning)
### Domain: Tindak Pidana Perdagangan Orang (TPPO)

Tujuan:
- Mengumpulkan 50 putusan Mahkamah Agung RI terkait TPPO
- Mengonversi putusan ke teks bersih
- Menyimpan hasil ke folder data/raw/

1.Import Library

In [28]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import os
import time

In [29]:
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

print("Browser siap (anti 403)")

Browser siap (anti 403)


In [30]:
BASE_URL = "https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdagangan-orang-{}.html"
TARGET_DATA = 50
MAX_PAGE = 10

RAW_DIR = "../data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

In [31]:
putusan_urls = []

for page in range(1, MAX_PAGE + 1):
    print(f"Membuka halaman {page}")
    driver.get(BASE_URL.format(page))

    try:
        # TUNGGU SAMPAI LIST PUTUSAN MUNCUL
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div#main-content")
            )
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # SELECTOR KHUSUS KARTU PUTUSAN
        cards = soup.select("div.spost.clearfix")

        print(f"  Ditemukan {len(cards)} kartu putusan")

        for card in cards:
            a = card.find("a", href=True)
            if a:
                url = a["href"]
                if not url.startswith("http"):
                    url = "https://putusan3.mahkamahagung.go.id" + url

                if url not in putusan_urls:
                    putusan_urls.append(url)

        print("  Total URL terkumpul:", len(putusan_urls))

        if len(putusan_urls) >= TARGET_DATA:
            break

        time.sleep(3)

    except Exception as e:
        print("  Gagal:", e)

Membuka halaman 1
  Gagal: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7a707fd95+15395]
	chromedriver!GetHandleVerifier [0x7ff7a707fdf0+153f0]
	chromedriver!(No symbol) [0x7ff7a6bb78fd]
	chromedriver!(No symbol) [0x7ff7a6c11479]
	chromedriver!(No symbol) [0x7ff7a6c1177c]
	chromedriver!(No symbol) [0x7ff7a6c61ef7]
	chromedriver!(No symbol) [0x7ff7a6c5eaeb]
	chromedriver!(No symbol) [0x7ff7a6c03a58]
	chromedriver!(No symbol) [0x7ff7a6c04953]
	chromedriver!GetHandleVerifier [0x7ff7a7645511+5dab11]
	chromedriver!GetHandleVerifier [0x7ff7a763f8fb+5d4efb]
	chromedriver!GetHandleVerifier [0x7ff7a7663205+5f8805]
	chromedriver!GetHandleVerifier [0x7ff7a709cc6e+3226e]
	chromedriver!GetHandleVerifier [0x7ff7a70a562c+3ac2c]
	chromedriver!GetHandleVerifier [0x7ff7a7089984+1ef84]
	chromedriver!GetHandleVerifier [0x7ff7a7089b14+1f114]
	chromedriver!GetHandleVerifier [0x7ff7a706ca77+2077]
	KERNEL32!BaseThreadInitThunk [0x7ffd7cdfe957+17]
	ntdll!RtlUserThreadStart [0x7ffd7e44427c+2c]

M

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff7a707fd95+15395]
	chromedriver!GetHandleVerifier [0x7ff7a707fdf0+153f0]
	chromedriver!(No symbol) [0x7ff7a6bb76e6]
	chromedriver!(No symbol) [0x7ff7a6c02ad3]
	chromedriver!(No symbol) [0x7ff7a6c39c82]
	chromedriver!(No symbol) [0x7ff7a6c33aa2]
	chromedriver!(No symbol) [0x7ff7a6c33263]
	chromedriver!(No symbol) [0x7ff7a6b803e5]
	chromedriver!GetHandleVerifier [0x7ff7a7645511+5dab11]
	chromedriver!GetHandleVerifier [0x7ff7a763f8fb+5d4efb]
	chromedriver!GetHandleVerifier [0x7ff7a7663205+5f8805]
	chromedriver!GetHandleVerifier [0x7ff7a709cc6e+3226e]
	chromedriver!GetHandleVerifier [0x7ff7a70a562c+3ac2c]
	chromedriver!(No symbol) [0x7ff7a6b7f188]
	chromedriver!GetHandleVerifier [0x7ff7a7801238+796838]
	KERNEL32!BaseThreadInitThunk [0x7ffd7cdfe

In [ ]:
putusan_urls = putusan_urls[:TARGET_DATA]
print("Final URL:", len(putusan_urls))

In [ ]:
for i, url in enumerate(putusan_urls, start=1):
    print(f"Scraping {i}/50")
    driver.get(url)
    time.sleep(5)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    text = soup.get_text(separator="\n")

    with open(f"{RAW_DIR}/case_{i:03d}.txt", "w", encoding="utf-8") as f:
        f.write(text)

In [ ]:
driver.quit()
print("Scraping selesai")